# Read yahoo history

In [1]:
import requests
from pandas import json_normalize
import pandas as pd
import datetime

/Users/hochakkong/Documents/github/finalproject/bootcamp-env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## setting

In [ ]:
now = datetime.datetime.now()
res = int(now.timestamp() * 1000) #end unix date

stockLst = [
        "TSLA", "NVDA", "AAPL", "META", "GOOGL",
        "MSFT", "INTC", "AVGO", "SOUN", "AMD",
        "NFLX", "UBER", "TGT", "BBAI", "OSS"
        ,"DNUT", "COMP", "OPEN", "MARA", "IOVA"
        ,"WMT",  "HD", "AMZN", "NIO", "COST"
        ,"PYPL", "ADBE", "CRM", "ORCL", "IBM"
        ,"CAH", "ACGL", "BLL", "CEGVV", "CLX"
        ,"CL", "LMT", "CPRT", "CAH", "MDLZ"
        ,"FANG",  "EL", "EXC", "LOW", "MAR"
        ,"GILD", "HAL", "LMT", "MCD", "NKE"
        ,"LHX", "MET", "MO", "MS", "V"
        ,"LLY", "ORCL", "JNJ", "PG", "KO"
        ,"TSM", "PDD", "CRWV"
      ]

start_unix_date = "1657237004"
end_unix_date = res

headers = {
    "User-Agent": "Mozilla/5.0"
}

# tslaUrl = r'https://query1.finance.yahoo.com/v8/finance/chart/NVDA?period1=1657237004&period2=1751931404&interval=1d&events=history'
# tslaUrl = '/content/sample_data/TSLA.json'

In [4]:
def stockUrl(stock):
  return f'https://query1.finance.yahoo.com/v8/finance/chart/{stock}?period1={start_unix_date}&period2={end_unix_date}&interval=1d&events=history'

def dfTreatment(url, headers):
  response = requests.get(url, headers = headers)
  data = response.json()
  if (data['chart']['result'] is not None):
    df_metaData = json_normalize(data['chart']['result'])
    df_indicator = pd.DataFrame(df_metaData['indicators.quote'][0])
    return df_metaData, df_indicator
  else:
    return None, None

def dfExtract(df_metaData, df_indicator, dfStock, dfAll, i):
  if df_metaData is not None:
    for j in df_indicator.columns.to_list():
      dfStock[j] = df_indicator[j][0]
    dfStock['code'] = i
    dfStock['datetime'] = df_metaData['timestamp'][0]
    #print(dfStock.columns.to_list())
  return dfStock

def dfConact(dfAll, dfStock):
  if dfAll.size == 0:
    dfAll=pd.DataFrame(columns=dfStock.columns.to_list())

  dfAll = pd.concat([dfAll, dfStock], ignore_index = True)
  return dfAll

def dfFillNA(dfAll):
  if dfAll.isna().any().any() == "np.True_":
    dfAll = dfAll.fillna(method = "ffill")
    msg = 'NA value'
  else:
    msg = 'no NA'
  # dfAll.to_csv("OLHC.csv")
  return dfAll, msg


In [5]:
dfAll = pd.DataFrame()

for i in stockLst:

  url = stockUrl(i)

  df_3, df_4 = dfTreatment(url, headers)
  
  dfStock = pd.DataFrame()

  dfStock = dfExtract(df_3, df_4, dfStock, dfAll,i)

  dfAll = dfConact(dfAll, dfStock)

dfAll['datetime'] = pd.to_datetime(dfAll['datetime'], unit="s").dt.date
dfAll, msg = dfFillNA(dfAll)
dfAll = dfAll.sort_values(by=['code', 'datetime'])

print(f"Completed! {msg}" )

/var/folders/3x/v2v5lgj97jqg15bcfj_8xq5m0000gn/T/ipykernel_42077/4133449056.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dfAll = pd.concat([dfAll, dfStock], ignore_index = True)


Completed! no NA


In [6]:
dfAll

,high,low,open,close,volume,code,datetime
1594,146.550003,143.279999,143.289993,146.350006,66253700,AAPL,2022-07-07
1595,147.550003,145.000000,145.259995,147.039993,64547800,AAPL,2022-07-08
1596,146.639999,143.779999,145.669998,144.869995,63141600,AAPL,2022-07-11
1597,148.449997,145.050003,145.759995,145.860001,77588800,AAPL,2022-07-12
1598,146.449997,142.119995,142.990005,145.490005,71185600,AAPL,2022-07-13
...,...,...,...,...,...,...,...
16732,99.440002,97.500000,97.849998,99.440002,18995900,WMT,2025-09-03
16733,101.349998,99.650002,99.910004,100.930000,15640900,WMT,2025-09-04
16734,101.480003,99.550003,101.080002,100.510002,13748500,WMT,2025-09-05
16735,102.360001,100.419998,100.849998,102.279999,13685700,WMT,2025-09-08


## df_logo

In [ ]:
df_logo = pd.read_csv("Stock_List.csv")
df_logo["symbol_link"] = "https://static2.finnhub.io/file/publicdatany/finnhubimage/stock_logo/"+ df_logo["symbol"]+".png"
# df_logo["symbol_link"] = "https://eodhd.com/img/logos/US/"+ df_logo["symbol"]+".png"

In [8]:
dfAll = dfAll.merge(df_logo, left_on='code', right_on='symbol')
dfAll.drop(['symbol'], axis = 1, inplace = True)
dfAll['index'] = dfAll.index
dfAll

,high,low,open,close,volume,code,datetime,symbol_link,index
0,146.550003,143.279999,143.289993,146.350006,66253700,AAPL,2022-07-07,https://static2.finnhub.io/file/publicdatany/f...,0
1,147.550003,145.000000,145.259995,147.039993,64547800,AAPL,2022-07-08,https://static2.finnhub.io/file/publicdatany/f...,1
2,146.639999,143.779999,145.669998,144.869995,63141600,AAPL,2022-07-11,https://static2.finnhub.io/file/publicdatany/f...,2
3,148.449997,145.050003,145.759995,145.860001,77588800,AAPL,2022-07-12,https://static2.finnhub.io/file/publicdatany/f...,3
4,146.449997,142.119995,142.990005,145.490005,71185600,AAPL,2022-07-13,https://static2.finnhub.io/file/publicdatany/f...,4
...,...,...,...,...,...,...,...,...,...
46221,99.440002,97.500000,97.849998,99.440002,18995900,WMT,2025-09-03,https://static2.finnhub.io/file/publicdatany/f...,46221
46222,101.349998,99.650002,99.910004,100.930000,15640900,WMT,2025-09-04,https://static2.finnhub.io/file/publicdatany/f...,46222
46223,101.480003,99.550003,101.080002,100.510002,13748500,WMT,2025-09-05,https://static2.finnhub.io/file/publicdatany/f...,46223
46224,102.360001,100.419998,100.849998,102.279999,13685700,WMT,2025-09-08,https://static2.finnhub.io/file/publicdatany/f...,46224


# pandas write to DB

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import Table, Column, MetaData, Integer, String, Date, Float, BigInteger
from sqlalchemy.engine import URL
from dotenv import load_dotenv

load_dotenv()
local_db_password = os.getenv("LOCAL_DB_PASSWORD")
if not local_db_password:
  raise RuntimeError("LOCAL_DB_PASSWORD must be set in the environment or .env file")

engineLst = [URL.create(
  "postgresql+psycopg2",
  username="postgres",
  password=local_db_password,
  host="localhost",
  port=5532,
  database="bootcamp_2504",
)]

for engine in engineLst:
  bootcamp_engine = create_engine(engine)

  metadata = MetaData()

  ohlc_data_schema = Table(
    "ohlc_data", metadata,
    Column("id", BigInteger, primary_key=True),
    Column("code", String(10), nullable=False),
    Column("date", Date, nullable=False),
    Column("open", Float, nullable=False),
    Column("high", Float, nullable=False),
    Column("low", Float, nullable=False),
    Column("close", Float, nullable=False),
    Column("volume", BigInteger, nullable=False),
    Column("symbolink", String, nullable=False)
  )

  metadata.create_all(bootcamp_engine)
  dfAll.to_sql("ohlc_data", bootcamp_engine, index=False, if_exists="replace")
  print("OK: historical data written")

  metadata = MetaData()

  stock_logo_schema = Table(
    "stock_logo", metadata,
    Column("symbol", String(15), primary_key=True),
    Column("logo_link", String(100), nullable=True)
  )
  metadata.create_all(bootcamp_engine)
  df_logo.to_sql("stock_logo", bootcamp_engine, index=False, if_exists="replace")

In [ ]:
df_logo